# SILVER

In [1]:
import duckdb

In [2]:
con = duckdb.connect(database = "dados_duckdb.db", read_only=False)

In [5]:
df = con.execute("SELECT * FROM bronze_z0019").fetch_df()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-01-11 17:57:43.739535
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-01-11 17:57:43.739535
2,10003,PORCA,BT10,100,100,z0019_1.csv,2026-01-11 17:57:43.739535
3,10004,ARRUELA,BT10,100,1000,z0019_1.csv,2026-01-11 17:57:43.739535
4,10005,PREGO,BT10,100,100,z0019_1.csv,2026-01-11 17:57:43.739535
5,10006,BUCHA,BT10,100,1500,z0019_1.csv,2026-01-11 17:57:43.739535
6,10007,REBITE,BT10,100,2000,z0019_1.csv,2026-01-11 17:57:43.739535
7,10008,CHAVE,BT10,100,300,z0019_1.csv,2026-01-11 17:57:43.739535
8,10009,PARAFUSADEIRA,BT10,100,100,z0019_1.csv,2026-01-11 17:57:43.739535
9,10010,ALICATE,BT10,100,100,z0019_1.csv,2026-01-11 17:57:43.739535


In [14]:
df = (con
      .execute("""
                SELECT *
                FROM (
                         SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row
                         FROM bronze_z0019
                         WHERE data_ingestao >= '2025-01-11'
                ) WHERE row = 1
               """).fetch_df()
      )

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10010,ALICATE,BT10,100,100,z0019_1.csv,2026-01-11 17:57:43.739535,1
1,10013,ESQUADRO,BT10,100,400,z0019_2.csv,2026-01-11 17:58:21.701522,1
2,10012,NIVEL,BT10,100,100,z0019_1.csv,2026-01-11 17:57:43.739535,1
3,10015,PONTEIRO,BT10,100,150,z0019_2.csv,2026-01-11 17:58:21.701522,1
4,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-01-11 17:57:43.739535,1
5,10011,TRENA,BT10,100,100,z0019_1.csv,2026-01-11 17:57:43.739535,1
6,10003,PORCA,BT10,100,100,z0019_1.csv,2026-01-11 17:57:43.739535,1
7,10016,LIXA,BT10,100,2000,z0019_2.csv,2026-01-11 17:58:21.701522,1
8,10008,CHAVE,BT10,100,300,z0019_1.csv,2026-01-11 17:57:43.739535,1
9,10014,TALHADEIRA,BT10,100,200,z0019_2.csv,2026-01-11 17:58:21.701522,1


In [18]:
df_final = (df.drop(columns=['nome_arquivo', 'data_ingestao', 'row']))
df_final = df_final.rename(columns={'NATBR': 'id'})
df_final = df_final.rename(columns={'MAKTX': 'nm_produto'})
df_final = df_final.rename(columns={'WERKS': 'id_categoria'})
df_final = df_final.rename(columns={'MAINS': 'id_fornecedor'})
df_final = df_final.rename(columns={'LABST': 'vl_preco'})

df_final.head(30)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10010,ALICATE,BT10,100,100
1,10013,ESQUADRO,BT10,100,400
2,10012,NIVEL,BT10,100,100
3,10015,PONTEIRO,BT10,100,150
4,10001,PARAFUSO,BT10,100,100
5,10011,TRENA,BT10,100,100
6,10003,PORCA,BT10,100,100
7,10016,LIXA,BT10,100,2000
8,10008,CHAVE,BT10,100,300
9,10014,TALHADEIRA,BT10,100,200


In [19]:
df_final.dtypes

id               object
nm_produto       object
id_categoria     object
id_fornecedor    object
vl_preco         object
dtype: object

In [31]:
df2 = df_final.astype(
    {
        'id': 'int32',
        'nm_produto': 'string',
        'id_categoria': 'string',
        'id_fornecedor': 'int32',
        'vl_preco': 'float32'
    }
)
df2.dtypes


id                        int32
nm_produto       string[python]
id_categoria     string[python]
id_fornecedor             int32
vl_preco                float32
dtype: object

In [32]:
con.execute("""
            CREATE TABLE IF NOT EXISTS produtos (
                    id BIGINT,
                    nm_produto TEXT,
                    id_categoria TEXT,
                    id_fornecedor BIGINT,
                    vl_preco FLOAT)
""")

In [33]:
df2.head()

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10010,ALICATE,BT10,100,100.0
1,10013,ESQUADRO,BT10,100,400.0
2,10012,NIVEL,BT10,100,100.0
3,10015,PONTEIRO,BT10,100,150.0
4,10001,PARAFUSO,BT10,100,100.0


In [35]:
con.execute("INSERT INTO produtos SELECT * FROM df2")

In [37]:
df_resultado = con.execute("SELECT * FROM produtos").fetch_df()
df_resultado.head(30)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10010,ALICATE,BT10,100,100.0
1,10013,ESQUADRO,BT10,100,400.0
2,10012,NIVEL,BT10,100,100.0
3,10015,PONTEIRO,BT10,100,150.0
4,10001,PARAFUSO,BT10,100,100.0
5,10011,TRENA,BT10,100,100.0
6,10003,PORCA,BT10,100,100.0
7,10016,LIXA,BT10,100,2000.0
8,10008,CHAVE,BT10,100,300.0
9,10014,TALHADEIRA,BT10,100,200.0


In [39]:
con.execute("COMMIT") # Garante a persistência total no arquivo .db
con.close()

ConnectionException: Connection Error: Connection already closed!